# Deep Learning Model Comparison for Badminton Shot Classification

**Purpose:** Train and compare multiple deep learning architectures on the same dataset

**Models to Compare:**
1. **LSTM** (Baseline) - Expected: 75-82%
2. **ST-GCN** (Strong Baseline) - Expected: 85-90%
3. **CTR-GCN** (Production) - Expected: 87-92%
4. **MS-TCN** (Fast Alternative) - Expected: 82-88%

**Dataset:**
- 5 shot types: Smash, Clear, Drop, Lift, Drive
- ~23,531 clips from ShuttleSet dataset
- MediaPipe pose landmarks (33 keypoints)

**Comparison Metrics:**
- Test accuracy
- Per-class F1 scores
- Training time
- Inference speed
- Model parameters
- Confusion matrices

**Total time:** ~8-12 hours (trains all models sequentially)

---

## Part 1: Setup and Data Loading

### 1.1 Install Dependencies

In [ ]:
# Set matplotlib backend
import os
os.environ['MPLBACKEND'] = 'Agg'

# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q scikit-learn pandas numpy tqdm matplotlib seaborn
!pip install -q tensorboard

print("✓ Dependencies installed")

### 1.2 Verify GPU and Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import time
from datetime import datetime
import json

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("\n✓ Libraries imported and seeds set")

### 1.3 Download Data from GCS

**Note:** This assumes pose extraction is complete and data is in GCS

In [ ]:
# Create directories
!mkdir -p data/processed/poses
!mkdir -p outputs/models
!mkdir -p outputs/reports

# Download pose data
print("Downloading pose data from GCS...")
!gsutil -m rsync -r gs://iti123storage/features/poses/ data/processed/poses/

# Download metadata
!gsutil cp gs://iti123storage/data/metadata.csv data/metadata.csv

print("\n✓ Data downloaded")

### 1.4 Load and Validate Metadata

In [ ]:
# Load metadata
df = pd.read_csv('data/metadata.csv')

# Filter for 5 target classes
TARGET_CLASSES = ['smash', 'clear', 'drop', 'lift', 'drive']
df['stroke_type'] = df['stroke_type'].str.lower()
df = df[df['stroke_type'].isin(TARGET_CLASSES)].copy()

print(f"{'='*60}")
print("DATASET OVERVIEW")
print(f"{'='*60}")
print(f"Total samples: {len(df):,}")
print(f"\nClass distribution:")
for shot_type in TARGET_CLASSES:
    count = len(df[df['stroke_type'] == shot_type])
    pct = count / len(df) * 100
    print(f"  {shot_type.capitalize():10s}: {count:6,} ({pct:5.1f}%)")

# Check imbalance
counts = df['stroke_type'].value_counts()
imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 3.0:
    print("⚠️  High class imbalance - will use class weights")

print(f"{'='*60}")

### 1.5 Load Pose Sequences (with validation)

In [ ]:
print("Loading and validating pose sequences...")
print("This may take 10-20 minutes\n")

pose_sequences = []
labels = []
player_ids = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Loading"):
    pose_file = Path(row['pose_file'])
    if not pose_file.is_absolute():
        pose_file = Path('data/processed/poses') / pose_file.name
    
    if not pose_file.exists():
        continue
    
    try:
        with open(pose_file, 'rb') as f:
            pose_data = pickle.load(f)
        
        # Validate shape
        if len(pose_data.shape) == 3 and pose_data.shape[1] == 33 and pose_data.shape[2] == 3:
            pose_sequences.append(pose_data)
            labels.append(row['stroke_type'])
            player_ids.append(row['player_id'])
            valid_indices.append(idx)
        
    except Exception as e:
        continue

labels = np.array(labels)
player_ids = np.array(player_ids)

print(f"\n✓ Loaded {len(pose_sequences):,} valid pose sequences")
print(f"Success rate: {len(pose_sequences)/len(df)*100:.1f}%")

### 1.6 Preprocess and Normalize (FIXED)

**IMPORTANT:** Proper normalization is critical for model performance!

In [ ]:
def normalize_pose(pose_sequence):
    """
    Normalize by torso center and body height
    
    CRITICAL FIX: This ensures models see relative movements, not absolute positions
    Without this, accuracy will be ~30% instead of ~85%
    """
    LEFT_HIP, RIGHT_HIP, NOSE = 23, 24, 0
    
    # Calculate hip center (torso reference point)
    hip_center = (pose_sequence[:, LEFT_HIP, :2] + pose_sequence[:, RIGHT_HIP, :2]) / 2
    
    # Center all keypoints by hip
    centered = pose_sequence.copy()
    centered[:, :, :2] = centered[:, :, :2] - hip_center[:, np.newaxis, :]
    
    # Calculate body height (nose to hip distance)
    body_heights = np.linalg.norm(pose_sequence[:, NOSE, :2] - hip_center, axis=1)
    body_height = np.mean(body_heights)
    
    # Scale by body height (normalizes for different player sizes)
    if body_height > 0.01:  # Avoid division by zero
        centered[:, :, :2] = centered[:, :, :2] / body_height
    
    return centered

def is_valid_single_person(pose_sequence, max_width=0.6):
    """
    Filter out multi-player detections
    
    CRITICAL: Badminton videos have 2 players. MediaPipe sometimes detects both,
    creating a "skeleton" that spans the entire court. This breaks normalization.
    
    Args:
        pose_sequence: (T, 33, 3) pose array
        max_width: Maximum width ratio (0.6 = 60% of frame)
    
    Returns:
        True if sequence represents a single player
    """
    x_coords = pose_sequence[:, :, 0]
    x_range = np.max(x_coords) - np.min(x_coords)
    
    # Single person shouldn't span more than 60% of frame width
    return x_range < max_width

def pad_sequence(sequence, max_length=90):
    """Pad or truncate to fixed length"""
    T, V, C = sequence.shape
    if T >= max_length:
        # Truncate from center to preserve motion
        start = (T - max_length) // 2
        return sequence[start:start+max_length]
    else:
        # Pad with zeros
        padded = np.zeros((max_length, V, C), dtype=sequence.dtype)
        padded[:T] = sequence
        return padded

# Filter short sequences
MIN_FRAMES = 30  # Minimum 1 second of motion
filtered_sequences = []
filtered_labels = []
filtered_players = []

print("Filtering short sequences...")
for seq, label, player in zip(pose_sequences, labels, player_ids):
    if len(seq) >= MIN_FRAMES:
        filtered_sequences.append(seq)
        filtered_labels.append(label)
        filtered_players.append(player)

print(f"Filtered {len(pose_sequences) - len(filtered_sequences)} short clips (< {MIN_FRAMES} frames)")

# Filter multi-player detections
print("\nFiltering multi-player detections...")
single_player_sequences = []
single_player_labels = []
single_player_ids = []

for seq, label, player in zip(filtered_sequences, filtered_labels, filtered_players):
    if is_valid_single_person(seq):
        single_player_sequences.append(seq)
        single_player_labels.append(label)
        single_player_ids.append(player)

print(f"Filtered {len(filtered_sequences) - len(single_player_sequences)} multi-player clips (x-range > 60%)")

# Determine target length
lengths = [len(seq) for seq in single_player_sequences]
TARGET_LENGTH = min(90, max(lengths))
print(f"\nUsing target length: {TARGET_LENGTH} frames")

# Preprocess with normalization
print("Normalizing and padding sequences...\n")
X = []
for seq in tqdm(single_player_sequences, desc="Processing"):
    normalized = normalize_pose(seq)
    padded = pad_sequence(normalized, TARGET_LENGTH)
    X.append(padded)

X = np.array(X, dtype=np.float32)
labels = np.array(single_player_labels)
player_ids = np.array(single_player_ids)

print(f"\n✓ Preprocessed shape: {X.shape}")
print(f"  Format: (N, T, V, C) = (samples, time, vertices, channels)")

# Verify normalization
mean_val = X[:, :, :, :2].mean()
std_val = X[:, :, :, :2].std()
print(f"\nNormalization check:")
print(f"  Mean: {mean_val:.4f} (should be ~0.0)")
print(f"  Std: {std_val:.4f} (should be ~0.1-0.4)")

if abs(mean_val) > 0.1:
    print("⚠️  WARNING: Mean is not near 0 - normalization may have failed!")
if std_val < 0.05:
    print("⚠️  WARNING: Std too low - over-normalization!")
elif std_val > 0.6:
    print("⚠️  WARNING: Std still high - may have remaining multi-player clips")
else:
    print("✓ Normalization quality: Good")

# Summary
print(f"\n{'='*60}")
print("PREPROCESSING SUMMARY")
print(f"{'='*60}")
print(f"Original samples: {len(pose_sequences):,}")
print(f"After short filter: {len(filtered_sequences):,} (-{len(pose_sequences)-len(filtered_sequences):,})")
print(f"After multi-player filter: {len(single_player_sequences):,} (-{len(filtered_sequences)-len(single_player_sequences):,})")
print(f"Final dataset: {len(X):,} samples")
print(f"Data retention: {len(X)/len(pose_sequences)*100:.1f}%")
print(f"{'='*60}")

print("\n✓ Preprocessing complete")

### 1.7 Encode Labels and Calculate Class Weights

In [ ]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(labels)
NUM_CLASSES = len(le.classes_)

print(f"{'='*60}")
print("LABEL ENCODING")
print(f"{'='*60}")
for i, label in enumerate(le.classes_):
    count = np.sum(y == i)
    print(f"  {i}: {label.capitalize():10s} - {count:,} ({count/len(y)*100:.1f}%)")

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f"\nClass weights:")
for label, weight in zip(le.classes_, class_weights):
    print(f"  {label.capitalize():10s}: {weight:.3f}")

print(f"{'='*60}")

### 1.8 Train-Val-Test Split (Player-Based)

In [ ]:
# Split: 70% train, 15% val, 15% test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=player_ids))

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=42)  # 0.176 * 0.85 ≈ 0.15
train_idx, val_idx = next(gss_val.split(
    X[train_val_idx], y[train_val_idx], groups=player_ids[train_val_idx]
))

train_idx = train_val_idx[train_idx]
val_idx = train_val_idx[val_idx]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"{'='*60}")
print("DATA SPLIT")
print(f"{'='*60}")
print(f"Train: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val:   {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")

# Verify no player leakage
train_players = set(player_ids[train_idx])
val_players = set(player_ids[val_idx])
test_players = set(player_ids[test_idx])

if len(train_players & val_players) == 0 and len(train_players & test_players) == 0:
    print("\n✓ No player leakage detected")
else:
    print("\n⚠️  Warning: Player leakage detected!")

print(f"{'='*60}")

### 1.9 Create PyTorch DataLoaders

In [ ]:
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        pose = self.X[idx]  # (T, V, C)
        label = self.y[idx]
        
        # Convert to GCN format: (C, T, V, M)
        pose_tensor = torch.FloatTensor(pose).permute(2, 0, 1).unsqueeze(-1)
        label_tensor = torch.LongTensor([label])[0]
        
        return pose_tensor, label_tensor

BATCH_SIZE = 32

train_dataset = PoseDataset(X_train, y_train)
val_dataset = PoseDataset(X_val, y_val)
test_dataset = PoseDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

**✅ Checkpoint 1: Data Preparation Complete**

---

## Part 2: Model Definitions

### 2.1 MediaPipe Skeleton Graph

In [ ]:
MEDIAPIPE_EDGES = [
    # Arms
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
    # Torso
    (11, 23), (12, 24), (23, 24),
    # Legs
    (23, 25), (25, 27), (27, 29), (27, 31),
    (24, 26), (26, 28), (28, 30), (28, 32),
]

def get_adjacency_matrix(edges, num_nodes=33):
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    for i, j in edges:
        A[i, j] = A[j, i] = 1
    A += np.eye(num_nodes)  # Self-loops
    return A

A = get_adjacency_matrix(MEDIAPIPE_EDGES)
print(f"✓ Adjacency matrix created: {A.shape}")

### 2.2 Model 1: LSTM Baseline

In [ ]:
class BidirectionalLSTM(nn.Module):
    """Bidirectional LSTM - Expected: 75-82%"""
    def __init__(self, input_size=99, hidden_size=128, num_layers=2, num_classes=5, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        # x: (N, C, T, V, M)
        N, C, T, V, M = x.shape
        x = x.squeeze(-1).permute(0, 2, 1, 3).reshape(N, T, -1)
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

print("✓ LSTM model defined")

### 2.3 Model 2: ST-GCN

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels, A):
        super().__init__()
        self.register_buffer('A', torch.FloatTensor(A))
        self.conv = nn.Conv2d(in_channels, out_channels, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        N, C, T, V = x.shape
        x = x.permute(0, 1, 3, 2)
        x = torch.matmul(self.A, x)
        x = x.permute(0, 1, 3, 2)
        return self.relu(self.bn(self.conv(x)))

class STGCN(nn.Module):
    """ST-GCN - Expected: 85-90%"""
    def __init__(self, in_channels=3, num_classes=5, A=None, dropout=0.5):
        super().__init__()
        if A is None:
            A = get_adjacency_matrix(MEDIAPIPE_EDGES)
        
        self.gcn1 = GraphConvolution(in_channels, 64, A)
        self.gcn2 = GraphConvolution(64, 128, A)
        self.gcn3 = GraphConvolution(128, 256, A)
        
        self.tcn1 = nn.Sequential(
            nn.Conv2d(64, 64, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Dropout(dropout)
        )
        self.tcn2 = nn.Sequential(
            nn.Conv2d(128, 128, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Dropout(dropout)
        )
        self.tcn3 = nn.Sequential(
            nn.Conv2d(256, 256, (9, 1), padding=(4, 0)),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout(dropout)
        )
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.squeeze(-1)
        
        x = self.gcn1(x)
        x = self.tcn1(x)
        x = self.gcn2(x)
        x = self.tcn2(x)
        x = self.gcn3(x)
        x = self.tcn3(x)
        
        x = self.pool(x).view(N, -1)
        return self.fc(x)

print("✓ ST-GCN model defined")

### 2.4 Model 3: MS-TCN

In [ ]:
class MS_TCN(nn.Module):
    """Multi-Stage TCN - Expected: 82-88%"""
    def __init__(self, in_channels=99, num_classes=5, num_stages=2, dropout=0.3):
        super().__init__()
        
        self.stage1 = nn.Sequential(
            nn.Conv1d(in_channels, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, 3, padding=2, dilation=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.stage2 = nn.Sequential(
            nn.Conv1d(128, 256, 3, padding=4, dilation=4),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256, 256, 3, padding=8, dilation=8),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # x: (N, C, T, V, M)
        N, C, T, V, M = x.shape
        x = x.squeeze(-1).permute(0, 2, 1, 3).reshape(N, T, -1)  # (N, T, C*V)
        x = x.permute(0, 2, 1)  # (N, C*V, T)
        
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

print("✓ MS-TCN model defined")

**✅ Checkpoint 2: Models Defined**

---

## Part 3: Training and Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in tqdm(loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = 100. * correct / total
    return total_loss / len(loader), acc, np.array(all_preds), np.array(all_labels)

def train_model(model, model_name, num_epochs=50, patience=10):
    """
    Train a model with early stopping
    
    FIXED: Learning rate increased from 0.0001 to 0.001
    This is critical for proper convergence!
    """
    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*60}")
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    
    # CRITICAL FIX: Learning rate 0.001 (was 0.0001)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
    
    best_val_acc = 0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_acc)  # Schedule based on val_acc (not loss)
        
        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            # Save best model
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'epoch': epoch
            }, f'outputs/models/{model_name}_best.pth')
            print(f"  ✓ New best model saved (val_acc: {val_acc:.2f}%)")
        else:
            patience_counter += 1
            print(f"  Patience: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"\n⚠️  Early stopping triggered at epoch {epoch+1}")
            break
    
    training_time = time.time() - start_time
    
    # Load best model
    checkpoint = torch.load(f'outputs/models/{model_name}_best.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"\n✓ Training complete in {training_time/60:.1f} minutes")
    print(f"✓ Best val accuracy: {best_val_acc:.2f}%")
    
    return model, history, training_time

print("✓ Training functions defined (with fixes)")

**✅ Checkpoint 3: Training Pipeline Ready**

---

## Part 4: Train All Models

### 4.1 Train LSTM (Baseline)

In [ ]:
lstm_model = BidirectionalLSTM(num_classes=NUM_CLASSES)
lstm_model, lstm_history, lstm_time = train_model(lstm_model, "LSTM", num_epochs=50)

### 4.2 Train ST-GCN

In [ ]:
stgcn_model = STGCN(num_classes=NUM_CLASSES, A=A)
stgcn_model, stgcn_history, stgcn_time = train_model(stgcn_model, "STGCN", num_epochs=50)

### 4.3 Train MS-TCN

In [ ]:
mstcn_model = MS_TCN(num_classes=NUM_CLASSES)
mstcn_model, mstcn_history, mstcn_time = train_model(mstcn_model, "MSTCN", num_epochs=50)

**✅ Checkpoint 4: All Models Trained**

---

## Part 5: Model Comparison & Evaluation

### 5.1 Evaluate on Test Set

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Evaluate all models
models_dict = {
    'LSTM': lstm_model,
    'ST-GCN': stgcn_model,
    'MS-TCN': mstcn_model
}

results = {}

print(f"\n{'='*60}")
print("TEST SET EVALUATION")
print(f"{'='*60}\n")

for name, model in models_dict.items():
    test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, criterion, device)
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    results[name] = {
        'test_acc': test_acc,
        'test_loss': test_loss,
        'f1_score': f1,
        'y_pred': y_pred,
        'y_true': y_true
    }
    
    print(f"{name}:")
    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  Test Loss: {test_loss:.4f}\n")

print(f"{'='*60}")

### 5.2 Comprehensive Comparison Table

In [ ]:
# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Measure inference time
def measure_inference_time(model, num_iterations=100):
    model.eval()
    sample_batch, _ = next(iter(test_loader))
    sample_batch = sample_batch.to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(sample_batch)
    
    # Measure
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = model(sample_batch)
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    end = time.time()
    
    return (end - start) / num_iterations * 1000  # ms per batch

# Create comparison table
comparison_data = []

for name in ['LSTM', 'ST-GCN', 'MS-TCN']:
    model = models_dict[name]
    params = count_parameters(model)
    inf_time = measure_inference_time(model)
    
    comparison_data.append({
        'Model': name,
        'Test Acc (%)': f"{results[name]['test_acc']:.2f}",
        'F1 Score': f"{results[name]['f1_score']:.4f}",
        'Parameters': f"{params:,}",
        'Inference (ms/batch)': f"{inf_time:.2f}",
        'Training Time (min)': f"{globals()[f'{name.lower().replace("-", "")}_time']/60:.1f}"
    })

comparison_df = pd.DataFrame(comparison_data)

print(f"\n{'='*80}")
print("MODEL COMPARISON")
print(f"{'='*80}")
print(comparison_df.to_string(index=False))
print(f"{'='*80}\n")

# Save table
comparison_df.to_csv('outputs/reports/model_comparison.csv', index=False)
print("✓ Comparison table saved")

### 5.3 Visualize Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

histories = {'LSTM': lstm_history, 'ST-GCN': stgcn_history, 'MS-TCN': mstcn_history}

# Training accuracy
for name, hist in histories.items():
    axes[0, 0].plot(hist['train_acc'], label=name)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Training Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation accuracy
for name, hist in histories.items():
    axes[0, 1].plot(hist['val_acc'], label=name)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Training loss
for name, hist in histories.items():
    axes[1, 0].plot(hist['train_loss'], label=name)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Training Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Validation loss
for name, hist in histories.items():
    axes[1, 1].plot(hist['val_loss'], label=name)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Validation Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/reports/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training curves saved")

### 5.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(models_dict.items()):
    cm = confusion_matrix(results[name]['y_true'], results[name]['y_pred'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_,
                ax=axes[idx])
    axes[idx].set_title(f'{name}\nAccuracy: {results[name]["test_acc"]:.2f}%')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('outputs/reports/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved")

### 5.5 Per-Class Performance

In [ ]:
print(f"\n{'='*80}")
print("PER-CLASS PERFORMANCE")
print(f"{'='*80}\n")

for name in ['LSTM', 'ST-GCN', 'MS-TCN']:
    print(f"\n{name}:")
    print(classification_report(
        results[name]['y_true'], 
        results[name]['y_pred'],
        target_names=[c.capitalize() for c in le.classes_],
        digits=4
    ))

print(f"{'='*80}")

### 5.6 Generate Summary Report

In [ ]:
# Find best model
best_model_name = max(results.items(), key=lambda x: x[1]['test_acc'])[0]
best_acc = results[best_model_name]['test_acc']

summary = f"""
{'='*80}
DEEP LEARNING MODEL COMPARISON SUMMARY
{'='*80}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Dataset
-------
Total samples: {len(X):,}
Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}
Classes: {', '.join([c.capitalize() for c in le.classes_])}

Models Compared
---------------
{comparison_df.to_string(index=False)}

Best Model: {best_model_name}
Best Test Accuracy: {best_acc:.2f}%

Key Findings
------------
1. {best_model_name} achieved the highest accuracy ({best_acc:.2f}%)
2. Accuracy improvement over LSTM baseline: {best_acc - results['LSTM']['test_acc']:.2f}%
3. All models outperformed random chance (20%)

Performance by Shot Type (Best Model: {best_model_name})
---------------------------------------------------
"""

# Add per-class performance for best model
y_true = results[best_model_name]['y_true']
y_pred = results[best_model_name]['y_pred']

for i, class_name in enumerate(le.classes_):
    class_mask = y_true == i
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask]) * 100
    summary += f"  {class_name.capitalize():10s}: {class_acc:.2f}%\n"

summary += f"""
Files Generated
---------------
- Model weights: outputs/models/*_best.pth
- Comparison table: outputs/reports/model_comparison.csv
- Training curves: outputs/reports/training_curves.png
- Confusion matrices: outputs/reports/confusion_matrices.png

Recommendations
---------------
1. Use {best_model_name} for production deployment
2. Consider ensemble of top 2 models for maximum accuracy
3. Fine-tune on sport-specific data if available

{'='*80}
"""

print(summary)

# Save summary
with open('outputs/reports/model_comparison_summary.txt', 'w') as f:
    f.write(summary)

print("\n✓ Summary saved to outputs/reports/model_comparison_summary.txt")

### 5.7 Upload Results to GCS

In [ ]:
# Upload models
print("Uploading models to GCS...")
!gsutil -m rsync -r outputs/models/ gs://iti123storage/models/deep_learning/

# Upload reports
print("Uploading reports to GCS...")
!gsutil -m rsync -r outputs/reports/ gs://iti123storage/outputs/reports/deep_learning/

print("\n✓ All results uploaded to GCS")

**✅ Checkpoint 5: Model Comparison Complete**

---

## 🎉 Model Comparison Complete!

### Summary of Results

All models have been trained and evaluated. Key deliverables:

✅ **3 models trained**: LSTM, ST-GCN, MS-TCN

✅ **Comprehensive comparison**: Test accuracy, F1 scores, inference speed, parameters

✅ **Visualizations**: Training curves, confusion matrices, per-class performance

✅ **Best model identified**: Ready for production deployment

✅ **All results saved**: Models, reports, and visualizations backed up to GCS

### Expected Results

Based on research:
- **LSTM**: 75-82% (baseline)
- **ST-GCN**: 85-90% (strong baseline)
- **MS-TCN**: 82-88% (fast alternative)

**Improvement over LSTM**: 7-15 percentage points

### Next Steps

1. **Deploy best model** to production
2. **Optional**: Train CTR-GCN for 87-92% accuracy (requires additional implementation)
3. **Optional**: Create ensemble of top 2 models for maximum accuracy
4. **Integrate** with badminton analysis application

---

**Total time:** ~8-12 hours (depending on GPU and dataset size)

**Congratulations! Deep learning model comparison complete. 🚀**